## NASA-IBM Lunar FM Generation notebook

This notebook:
1. Loads data from netcdf/txt files
2. Loads a trained Lunar FM checkpoint
3. Generates token predictions for target modalities
4. Automatically decodes tokens to images using the tokenizers
5. Creates visual comparisons (target vs generated vs difference) for each channel

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from omegaconf import DictConfig, OmegaConf
from scipy.ndimage import zoom as _zoom
from tokenizers import Tokenizer
from tqdm import tqdm

import ni_lfm
from ni_lfm.models.ni_lfm_generation import get_lfm_generation_model
from ni_lfm.utils.generation import (
    encode_sequence_batch,
    build_modality_info,
    build_tokenizers_from_modality_info,
    denormalize_data,
    filter_modality_info,
    get_dataloader_for_generation,
    load_ckpt_for_generation,
    prepare_model_inputs,
    sort_text,
)
from ni_lfm.utils.generation import process_tok_target
from ni_lfm.utils.tokenizer import EOS_TOKEN, decode_token_sequences

%config InlineBackend.figure_format = "retina"


# Run the notebook as if launched from the repo root, so repo-relative
# paths all resolve regardless of the notebook location.
REPO_ROOT = Path(ni_lfm.__file__).parent.parent
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")


LOW_RES = ("vis", "dtm", "aspect", "slope", "uv")
HIGH_RES = ("nac", "dtm_3m", "aspect_3m", "slope_3m")
TEXT = ("metadata", "static_maps")
RAW_MODALITIES = LOW_RES + HIGH_RES + TEXT
TOKENIZED_MODALITIES = {k: f"tok_{k}" for k in HIGH_RES + LOW_RES}


CMAP = "viridis"
DIFF_CMAP = "RdBu_r"
C_HIST_FILL = "#0F62FE"
C_HIST_EDGE = "#FFFFFF"
C_GRID = "#8D8D8D"


#### Parameters

In [ ]:
checkpoint = "debug_data/backbone/checkpoint.pt"
cfg_path = "debug_data/backbone/config.yaml"
data_parquet_file = "debug_data/debug_dataset/WAC_low_res_100k_sample_debug/subsampled_data.parquet"
data_root = "debug_data/debug_dataset/WAC_low_res_100k_sample_debug"
in_domains = ["dtm", "metadata"]
out_domains = ["vis"]
tokenizers_root = "debug_data/tokenizers"
num_samples = 2
seed = 42
device = "cpu"
detokenizer_steps = 25
decoding_steps = 1
temperature = 1.0
top_p = 0.8
top_k = 0

#### Helper functions

In [ ]:

def union_domains(elements: list[list[str]]) -> list[str]:
    """Sorted union of a list of domain lists."""
    return sorted(set().union(*(set(e) for e in elements)))


def define_modalities(cfg: DictConfig, in_domains: list[str], out_domains: list[str]):
    """Override input/output domains with command-line names and check if they are available in the config."""

    all_domains_data, all_domains = [], []

    # Model can only generate tokenized modalities, but we detokenize with them
    model_out_domains = []
    for d in out_domains:
        model_out_domains.append(TOKENIZED_MODALITIES.get(d, d))

    # Check if input modalities is a subset of training modalities
    assert set(in_domains).issubset(set(cfg.in_domains))
    all_domains_data.append(in_domains)
    all_domains.append(in_domains)
    print(f"\nOverriding input modalities: {in_domains}")

    # Check if output modalities is a subset of training modalities
    assert set(model_out_domains).issubset(set(cfg.out_domains))
    all_domains_data.append(out_domains)
    all_domains.append(model_out_domains)
    print(f"\nOverriding output modalities: {out_domains}")

    all_domains_data = union_domains(elements=all_domains_data)
    all_domains = union_domains(elements=all_domains)

    return all_domains_data, model_out_domains, all_domains


def create_comparison_plot(
    target: torch.Tensor,
    generated: torch.Tensor,
    modality: str,
    channel_idx: int,
    original: torch.Tensor,
    conditioning_modalities: list[str] | None = None,
    file_name: str | None = None,
):
    """Create a two-row comparison plot showing original vs generated and target vs generated.

    Args:
        target: Target data (tokenized reconstruction) - torch tensor or numpy array
        generated: Generated data (model output) - torch tensor or numpy array
        modality: Modality name (e.g., 'tok_slope@100m')
        channel_idx: Channel index to visualize
        original: Optional original untokenized image for two-row comparison
        conditioning_modalities: Optional list of input modalities used for conditioning
        file_name: name of the sample file to show on the plot title
    """
    target = target.cpu().numpy()
    generated = generated.cpu().numpy()
    original = original.cpu().numpy()

    if target.ndim == 4:  # (B, C, H, W)
        target_ch = target[0, channel_idx]
        generated_ch = generated[0, channel_idx]
        original_ch = original[0, channel_idx]
    elif target.ndim == 3:  # (C, H, W)
        target_ch = target[channel_idx]
        generated_ch = generated[channel_idx]
        original_ch = original[channel_idx]
    else:
        target_ch = target
        generated_ch = generated
        original_ch = original

    # Resize generated to match target if shapes differ (e.g. tokenizer trained on
    # different resolution than the dataset images)
    if generated_ch.shape != target_ch.shape:
        zoom_factors = (target_ch.shape[0] / generated_ch.shape[0],
                        target_ch.shape[1] / generated_ch.shape[1])
        # Use bilinear interpolation (order=1) for resizing
        generated_ch = np.asarray(_zoom(generated_ch, zoom_factors, order=1))

    # Calculate differences
    difference_gen_vs_target = generated_ch - target_ch

    # Two-row layout: [Original vs Generated] and [Target Reconstruction vs Generated]
    _, axes = plt.subplots(2, 4, figsize=(20, 10))

    # Resize original to match target if needed
    if original_ch.shape != target_ch.shape:
        zoom_factors = (target_ch.shape[0] / original_ch.shape[0],
                        target_ch.shape[1] / original_ch.shape[1])
        original_ch = np.asarray(_zoom(original_ch, zoom_factors, order=1))

    difference_gen_vs_original = generated_ch - original_ch

    # Determine common colormap range for all images
    vmin = min(original_ch.min(), target_ch.min(), generated_ch.min())
    vmax = max(original_ch.max(), target_ch.max(), generated_ch.max())

    # Row 1: Original vs Generated
    # Plot original
    im1 = axes[0, 0].imshow(original_ch, cmap=CMAP, vmin=vmin, vmax=vmax)
    axes[0, 0].set_title(f"Original (ch {channel_idx})")
    axes[0, 0].axis("off")
    plt.colorbar(im1, ax=axes[0, 0], fraction=0.046)

    # Plot generated
    im2 = axes[0, 1].imshow(generated_ch, cmap=CMAP, vmin=vmin, vmax=vmax)
    axes[0, 1].set_title(f"Generated (ch {channel_idx})")
    axes[0, 1].axis("off")
    plt.colorbar(im2, ax=axes[0, 1], fraction=0.046)

    # Plot difference (generated - original)
    diff_max_1 = max(abs(difference_gen_vs_original.min()), abs(difference_gen_vs_original.max()))
    im3 = axes[0, 2].imshow(difference_gen_vs_original, cmap=DIFF_CMAP, vmin=-diff_max_1, vmax=diff_max_1)
    axes[0, 2].set_title("Generated - Original")
    axes[0, 2].axis("off")
    plt.colorbar(im3, ax=axes[0, 2], fraction=0.046)

    # Plot histogram
    axes[0, 3].hist(difference_gen_vs_original.flatten(), bins=50, color=C_HIST_FILL, edgecolor=C_HIST_EDGE, alpha=0.7)
    axes[0, 3].set_title("Difference Histogram")
    axes[0, 3].set_xlabel("Difference Value")
    axes[0, 3].set_ylabel("Frequency")
    axes[0, 3].grid(True, color=C_GRID, alpha=0.3)

    # Row 2: Target (Reconstruction) vs Generated
    # Plot target (tokenizer reconstruction)
    im4 = axes[1, 0].imshow(target_ch, cmap=CMAP, vmin=vmin, vmax=vmax)
    axes[1, 0].set_title(f"Target (Reconstruction)\nChannel {channel_idx}")
    axes[1, 0].axis("off")
    plt.colorbar(im4, ax=axes[1, 0], fraction=0.046)

    # Plot generated
    im5 = axes[1, 1].imshow(generated_ch, cmap=CMAP, vmin=vmin, vmax=vmax)
    axes[1, 1].set_title(f"Generated\nChannel {channel_idx}")
    axes[1, 1].axis("off")
    plt.colorbar(im5, ax=axes[1, 1], fraction=0.046)

    # Plot difference (generated - target)
    diff_max_2 = max(abs(difference_gen_vs_target.min()), abs(difference_gen_vs_target.max()))
    im6 = axes[1, 2].imshow(difference_gen_vs_target, cmap=DIFF_CMAP, vmin=-diff_max_2, vmax=diff_max_2)
    axes[1, 2].set_title("Difference\n(Generated - Target)")
    axes[1, 2].axis("off")
    plt.colorbar(im6, ax=axes[1, 2], fraction=0.046)

    # Plot histogram
    axes[1, 3].hist(difference_gen_vs_target.flatten(), bins=50, color=C_HIST_FILL, edgecolor=C_HIST_EDGE, alpha=0.7)
    axes[1, 3].set_title("Difference Histogram")
    axes[1, 3].set_xlabel("Difference Value")
    axes[1, 3].set_ylabel("Frequency")
    axes[1, 3].grid(True, color=C_GRID, alpha=0.3)

    # Build informative title including file, crop, and conditioning info
    title_parts = [modality, f"Channel {channel_idx}"]
    if file_name:
        title_parts.insert(0, file_name)
    if conditioning_modalities:
        title_parts.append(f"Conditioned on: {', '.join(conditioning_modalities)}")
    plt.suptitle(" | ".join(title_parts), fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close()


def visualize_text_sequence(
    target_tokens: torch.Tensor,
    generated_tokens: torch.Tensor,
    text_tokenizer: Tokenizer,
    modality: str,
):
    """Visualize metadata as text comparison.

    Creates a text file showing:
    - Target metadata (decoded from tokens)
    - Generated metadata (decoded from tokens)
    - Token-level comparison
    """
    target_texts = decode_token_sequences(target_tokens, text_tokenizer)
    generated_texts = decode_token_sequences(generated_tokens, text_tokenizer)

    # Calculate token-level accuracy
    if target_tokens.ndim == 3:
        target_tokens = target_tokens[:, :, 0]
    if generated_tokens.ndim == 3:
        generated_tokens = generated_tokens[:, :, 0]

    # Compare tokens (ignoring padding)
    target_ids = target_tokens[0].cpu().tolist()
    generated_ids = generated_tokens[0].cpu().tolist()

    # Filter padding
    target_ids = [i for i in target_ids if i != 0]
    generated_ids = [i for i in generated_ids if i != 0]

    # Sort metadata tokens alphabetically
    target_text_sorted, target_tokens_sorted = sort_text(target_texts)
    generated_text_sorted, generated_tokens_sorted = sort_text(generated_texts)

    # Calculate accuracy: for each target token, check if at least one generated token matches it exactly
    # Convert generated to set for faster lookup
    generated_set = set(generated_tokens_sorted)

    # Count how many target tokens have at least one exact match in generated
    matches = sum(1 for target_token in target_tokens_sorted if target_token in generated_set)

    exact_match = target_text_sorted == generated_text_sorted

    output_text = []
    output_text.extend(("=" * 80, f"Metadata Sequence Comparison: {modality}", "=" * 80))
    output_text.extend((
        "",
        "TARGET (Original):",
        "-" * 80,
        target_texts[0],
        "",
        "GENERATED (Original):",
        "-" * 80,
        generated_texts[0],
        "",
        "TARGET (Sorted Alphabetically):",
        "-" * 80,
        target_text_sorted,
        "",
        "GENERATED (Sorted Alphabetically):",
        "-" * 80,
        generated_text_sorted,
        "",
        "-" * 80,
        f"Exact Match: {exact_match}",
        f"Target Pairs: {len(target_tokens_sorted)}",
        f"Generated Pairs: {len(generated_tokens_sorted)}",
        f"Common Pairs: {matches}",
        "=" * 80,
    ))

    print(output_text)


def prepare_targets(
    batch: dict,
    out_domains: list[str],
    tokenizers: torch.nn.ModuleDict,
    modality_info: dict,
    text_tokenizer: Tokenizer | None,
    common_input_size: int,
    device: torch.device | str,
    timesteps: int | None,
) -> dict[str, torch.Tensor]:
    """Prepare target tensors, handling both raw images and pre-tokenized data."""
    targets = {}

    for modality in out_domains:
        # Try both the modality name and the tokenized name
        batch_key = modality
        if modality not in batch:
            # tokenized modalities are not available in batch -> translating to raw modality
            batch_key = next((k for k in TOKENIZED_MODALITIES if TOKENIZED_MODALITIES[k] == modality), None)
            if batch_key is None or batch_key not in batch:
                print(f"Warning: '{batch_key}' not found in batch. Available keys: {list(batch.keys())}")
                continue

        if modality_info[modality]["type"] == "seq":
            if text_tokenizer is None:
                raise ValueError(f"Text tokenizer required for sequence modality {modality}")
            raw_tok = encode_sequence_batch(sequence=batch[batch_key],
                                             max_tokens=modality_info[modality]["max_tokens"],
                                             text_tokenizer=text_tokenizer)
        else:
            raw_tok = batch[batch_key].to(device)

        if modality in tokenizers:
            targets[modality] = process_tok_target(
                raw_tok=raw_tok,
                tok_module=tokenizers[modality],
                input_size=common_input_size,
                timesteps=timesteps,
            ).cpu()
        else:
            targets[modality] = raw_tok

    return targets


def plot_input_modalities(batch: dict, input_modalities: list[str], modality_info: dict):
    """Plot figure with image input modalities: rows = modalities, cols = channels.

    Args:
        batch: Raw batch dict from the dataloader.
        input_modalities: List of input modality names (raw, no tok_ prefix).
        modality_info: Full modality info dict for denormalization.
    """
    # Collect only image modalities that are present in the batch
    img_mods = [
        m for m in input_modalities
        if modality_info.get(m, {}).get("type", "img") == "img" and m in batch
    ]
    if not img_mods:
        return

    # Denormalize each modality and convert to (C, H, W) numpy
    mod_arrays: dict[str, np.ndarray] = {}
    for mod in img_mods:
        data_np = denormalize_data(batch[mod], mod, modality_info).cpu().numpy()
        if data_np.ndim == 4:
            data_np = data_np[0]        # (B, C, H, W) -> (C, H, W)
        elif data_np.ndim == 2:
            data_np = data_np[None]     # (H, W)       -> (1, H, W)
        mod_arrays[mod] = data_np

    num_rows = len(img_mods)
    num_cols = max(arr.shape[0] for arr in mod_arrays.values())

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 5 * num_rows), squeeze=False)

    for row_idx, mod in enumerate(img_mods):
        arr = mod_arrays[mod]
        num_channels = arr.shape[0]
        vmin, vmax = float(arr.min()), float(arr.max())

        for col_idx in range(num_cols):
            ax = axes[row_idx, col_idx]
            if col_idx < num_channels:
                im = ax.imshow(arr[col_idx], cmap=CMAP, vmin=vmin, vmax=vmax)
                ax.set_title(f"{mod}\nChannel {col_idx}")
                plt.colorbar(im, ax=ax, fraction=0.046)
            else:
                ax.set_visible(False)  # blank for modalities with fewer channels
            ax.axis("off")

    fig.suptitle("Input modalities", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close()


def get_visualizations(
        batch,
        targets,
        outputs,
        modality_info,
        text_tokenizer,
        input_modalities,
        file_name=None,
    ):

    plot_input_modalities(
        batch=batch,
        input_modalities=input_modalities,
        modality_info=modality_info,
    )

    for modality in outputs:

        if modality not in targets:
            print(f"Warning: No target for {modality}, skipping visualization...")
            continue

        target = targets[modality]
        generated = outputs[modality]

        if modality_info.get(modality, {}).get("type", "img") == "seq":
            visualize_text_sequence(
                target_tokens=target,
                generated_tokens=generated,
                text_tokenizer=text_tokenizer,
                modality=modality,
            )

        else:
            # Try to extract original from batch - Use parent domain if tokenized
            bare_modality = modality_info[modality].get("parent_domain", modality)

            original_data = batch[bare_modality]
            original_denorm = denormalize_data(original_data, bare_modality, modality_info)

            target_denorm = denormalize_data(target, modality, modality_info)
            generated_denorm = denormalize_data(generated, modality, modality_info)

            # Get number of channels — use min of target/generated to avoid index errors
            # when the tokenizer outputs fewer channels than the raw target
            target_channels = target_denorm.shape[1] if target_denorm.ndim == 4 else 1
            gen_channels = generated.shape[1] if generated.ndim == 4 else 1
            num_channels = min(target_channels, gen_channels)

            # Create visualization for each channel (continuous modalities)
            for ch_idx in range(num_channels):
                create_comparison_plot(
                    target=target_denorm,
                    generated=generated_denorm,
                    modality=modality,
                    channel_idx=ch_idx,
                    original=original_denorm,
                    conditioning_modalities=input_modalities,
                    file_name=file_name,
                )


#### Initializing...

In [ ]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


print(f"Loading training config from {cfg_path}")
train_cfg = OmegaConf.load(cfg_path)

device = torch.device(device)
print(f"\nUsing device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Load text tokenizer (path in the config is repo-relative; cwd is set to
# REPO_ROOT in the imports cell so it resolves directly).
print(f"Loading text tokenizer from {train_cfg.data.text_tokenizer_path}")
if train_cfg.data.text_tokenizer_path is None:
    text_tokenizer = None
    print("WARNING: No text tokenizer path in config, text_tokenizer will be None")
else:
    text_tokenizer = Tokenizer.from_file(train_cfg.data.text_tokenizer_path)
    print(f"Text tokenizer loaded successfully: {type(text_tokenizer)}")

all_domains_data, model_out_domains, all_domains = define_modalities(train_cfg.data, in_domains, out_domains)
print(f"\nAll modalities: {all_domains_data}")
print(f"All modalities (model): {all_domains}")


#### Preparing data loader

In [ ]:
modality_info, common_input_size = build_modality_info(train_cfg)

# Modality info needs raw modalities for dataloading and tokenized output modalities for model
dataset_mod_info = filter_modality_info(all_domains_data, modality_info)
model_mod_info = filter_modality_info(all_domains, modality_info)

dataloader = get_dataloader_for_generation(
    data_root=data_root,
    index_path=data_parquet_file,
    all_domains=all_domains_data,
    modality_info=dataset_mod_info,
    input_size=common_input_size,
    num_samples=num_samples,
    img_domains=[d for d in all_domains if d not in TEXT],
    return_file_name=True,
)

#### Initializing and loading model

In [ ]:

tokenized_modalities = [mod for mod in model_out_domains if mod not in TEXT]
tokenizers = build_tokenizers_from_modality_info(
    modality_info=model_mod_info,
    modalities=tokenized_modalities,
    tokenizers_root=tokenizers_root,
)
print(f"Tokenizers loaded: {list(tokenizers.keys())}")
print(f"Creating model: {train_cfg.model.name}")

model = get_lfm_generation_model(
    variant=train_cfg.model.name,
    input_modalities=in_domains,
    output_modalities=model_out_domains,
    modality_info=model_mod_info,
    pretrained_tokenizers=tokenizers,
    text_tokenizer=text_tokenizer,
    cfg=train_cfg,
    num_register_tokens=train_cfg.model.num_register_tokens,
    timesteps=detokenizer_steps,
    decoding_steps=decoding_steps,
    temps=temperature,
    top_p=top_p,
    top_k=top_k,
)

load_ckpt_for_generation(checkpoint, model)

model = model.to(device)
model.eval()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


#### Running generation and plotting results

In [ ]:
with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Processing batches")):
        print(f"\nBatch {batch_idx} keys: {list(batch.keys())}")

        model_inputs = prepare_model_inputs(
            batch=batch,
            in_domains=in_domains,
            modality_info=modality_info,
            text_tokenizer=text_tokenizer,
            device=device,
        )

        if not model_inputs:
            print(f"Warning: No valid inputs in batch {batch_idx}, skipping...")
            continue

        targets = prepare_targets(
            batch=batch,
            out_domains=model_out_domains,
            tokenizers=model.tokenizers,
            modality_info=modality_info,
            text_tokenizer=text_tokenizer,
            common_input_size=common_input_size,
            device=device,
            timesteps=detokenizer_steps,
        )

        outputs = model(model_inputs)

        print(f"Outputs keys: {list(outputs.keys())}")
        print(f"Targets keys: {list(targets.keys())}")

        for mod in [k for k in outputs if modality_info.get(k, {}).get("type") == "seq"]:
            if not isinstance(outputs[mod], torch.Tensor):
                outputs[mod] = torch.tensor(
                    np.array([text_tokenizer.token_to_id(x) for x in outputs[mod] if x != ""])[None],
                    dtype=torch.int32,
                )
            else:
                outputs[mod] = outputs[mod].cpu()

            idx = ((targets[mod][0] == text_tokenizer.token_to_id(EOS_TOKEN)) | (targets[mod][0] == 0)).nonzero(as_tuple=True)[0]
            if idx.numel() > 0:
                targets[mod] = targets[mod][:, :idx[0]]

        # Create visualizations for each output modality
        get_visualizations(
            batch=batch,
            targets=targets,
            outputs=outputs,
            modality_info=modality_info,
            text_tokenizer=text_tokenizer,
            input_modalities=in_domains,
            file_name=batch["file_name"][0] if "file_name" in batch else None,
        )